# Module 08 -- Exotic Payoffs

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

Exotics are where options get interesting -- and where the real money
is made and lost. Barrier options, digitals, and path-dependent
structures are the bread and butter of FX and structured products desks.

I traded these for years. The first thing you learn is that the
closed-form price is the easy part. The hard part is the risk you
cannot see in the Greeks -- the discontinuities, the gap risk,
and the fact that your counterparty knows exactly where your barrier is.

---
*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*


In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt


## Barrier Options -- Down-and-Out Call

A down-and-out call is a regular call that **dies** (knocks out) if
the spot touches a barrier $B$ below the current price. You get the
upside, but if the market dips to $B$ first, the option is worthless.

The closed-form is from Merton (1973) / Reiner-Rubinstein (1991).
It looks ugly but it is just a combination of vanilla BSM terms
with power-law adjustments for the barrier.


In [ ]:
def down_and_out_call(S, K, B, T, r, sigma, q=0):
    """
    Down-and-out call: knocked out if S touches B (B < S, B < K).
    Closed-form under Black-Scholes assumptions.
    """
    if S <= B:
        return 0.0  # already knocked out

    lam = (r - q + 0.5 * sigma**2) / (sigma**2)
    _x1 = np.log(S / K) / (sigma * np.sqrt(T)) + lam * sigma * np.sqrt(T)
    y1 = np.log(B**2 / (S * K)) / (sigma * np.sqrt(T)) + lam * sigma * np.sqrt(T)

    # Vanilla call
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    vanilla = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    # Barrier adjustment
    barrier_adj = S * np.exp(-q * T) * (B / S)**(2 * lam) * norm.cdf(y1) \
                - K * np.exp(-r * T) * (B / S)**(2 * lam - 2) * norm.cdf(
                    y1 - sigma * np.sqrt(T))

    return vanilla - barrier_adj


# Quick test
S, K, B = 100, 105, 90
T, r, sigma = 0.5, 0.05, 0.25

doc_price = down_and_out_call(S, K, B, T, r, sigma)
print(f"Down-and-out call: {doc_price:.4f}")
print(f"Vanilla call:      {down_and_out_call(S, K, 0.01, T, r, sigma):.4f}")
print(f"Barrier discount:  {1 - doc_price / down_and_out_call(S, K, 0.01, T, r, sigma):.1%}")


## Up-and-In Put

An up-and-in put only **activates** if the spot rises to the
barrier $B$ first. Think of it as cheap downside protection that
only kicks in after a rally -- useful in structured products where
the client wants crash protection but does not want to pay full
put premium.


In [ ]:
def up_and_in_put(S, K, B, T, r, sigma, q=0):
    """
    Up-and-in put: activates only if S touches B (B > S).
    Uses in-out parity: up-in put = vanilla put - up-out put.
    For B >= K, simplified closed-form.
    """
    if S >= B:
        # Already hit the barrier -- it is a vanilla put
        d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)

    lam = (r - q + 0.5 * sigma**2) / (sigma**2)
    y = np.log(B**2 / (S * K)) / (sigma * np.sqrt(T)) + lam * sigma * np.sqrt(T)
    _y1 = np.log(B / S) / (sigma * np.sqrt(T)) + lam * sigma * np.sqrt(T)

    uip = -S * np.exp(-q * T) * (B / S)**(2 * lam) * norm.cdf(-y + sigma * np.sqrt(T)) \
          + K * np.exp(-r * T) * (B / S)**(2 * lam - 2) * norm.cdf(-y)

    return max(uip, 0)


uip_price = up_and_in_put(S=100, K=95, B=110, T=0.5, r=0.05, sigma=0.25)
print(f"Up-and-in put (K=95, B=110): {uip_price:.4f}")


## Barrier Sensitivity -- The Cliff

This is what you need to see. The price of a barrier option has a
discontinuity at the barrier. The closer spot gets to the barrier,
the more violent the Greeks become.


In [ ]:
spots = np.linspace(92, 115, 300)
barrier_up = 110
barrier_down = 92

doc_prices = [down_and_out_call(s, 105, barrier_down, 0.5, 0.05, 0.25) for s in spots]
uip_prices = [up_and_in_put(s, 95, barrier_up, 0.5, 0.05, 0.25) for s in spots]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(spots, doc_prices, 'b-', linewidth=2)
axes[0].axvline(barrier_down, color='red', linewidth=2, linestyle='--', label=f'Barrier ({barrier_down})')
axes[0].set_title('Down-and-Out Call (K=105, B=92)')
axes[0].set_xlabel('Spot')
axes[0].set_ylabel('Option Price')
axes[0].legend()

axes[1].plot(spots, uip_prices, 'r-', linewidth=2)
axes[1].axvline(barrier_up, color='blue', linewidth=2, linestyle='--', label=f'Barrier ({barrier_up})')
axes[1].set_title('Up-and-In Put (K=95, B=110)')
axes[1].set_xlabel('Spot')
axes[1].set_ylabel('Option Price')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/08_barrier_prices.png', dpi=100, bbox_inches='tight')
plt.show()


## Digital (Binary) Options

A digital call pays a fixed amount $Q$ if spot finishes above $K$,
zero otherwise. It is the building block of structured products.
The replication is simple: it is the limit of a tight call spread.

In practice, nobody hedges a digital with the closed-form delta.
You hedge it as a call spread and manage the residual. The delta
of a digital near expiry near the strike is a nightmare -- it goes
to infinity. That is not a theoretical problem, it is a real P&L
problem I have seen blow up a junior trader's book.


In [ ]:
def digital_call(S, K, T, r, sigma, Q=1.0, q=0):
    """Cash-or-nothing digital call paying Q if S_T > K."""
    d2 = (np.log(S / K) + (r - q - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return Q * np.exp(-r * T) * norm.cdf(d2)


def digital_put(S, K, T, r, sigma, Q=1.0, q=0):
    """Cash-or-nothing digital put paying Q if S_T < K."""
    d2 = (np.log(S / K) + (r - q - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return Q * np.exp(-r * T) * norm.cdf(-d2)


dc = digital_call(100, 100, 0.25, 0.05, 0.20, Q=1)
dp = digital_put(100, 100, 0.25, 0.05, 0.20, Q=1)
print(f"Digital call (ATM, Q=1): {dc:.4f}")
print(f"Digital put  (ATM, Q=1): {dp:.4f}")
print(f"Sum (should ~ exp(-rT)): {dc + dp:.4f} vs {np.exp(-0.05*0.25):.4f}")


## Payoff Diagrams -- All Four


In [ ]:
spots_fine = np.linspace(80, 120, 500)
K_barrier = 100

# Payoffs at expiry (or just before, to show the smooth price)
T_plot = 5 / 365  # 5 days to expiry for smooth curves

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Down-and-out call
doc_pnl = [down_and_out_call(s, 100, 90, T_plot, 0.05, 0.25) for s in spots_fine]
axes[0, 0].plot(spots_fine, doc_pnl, 'b-', linewidth=2)
axes[0, 0].axvline(90, color='red', linestyle='--', linewidth=1.5, label='Barrier 90')
axes[0, 0].set_title('Down-and-Out Call (K=100, B=90)')
axes[0, 0].legend()
axes[0, 0].set_ylabel('Price')

# Up-and-in put
uip_pnl = [up_and_in_put(s, 100, 110, T_plot, 0.05, 0.25) for s in spots_fine]
axes[0, 1].plot(spots_fine, uip_pnl, 'r-', linewidth=2)
axes[0, 1].axvline(110, color='blue', linestyle='--', linewidth=1.5, label='Barrier 110')
axes[0, 1].set_title('Up-and-In Put (K=100, B=110)')
axes[0, 1].legend()

# Digital call payoff
dc_vals = [digital_call(s, 100, T_plot, 0.05, 0.25) for s in spots_fine]
dc_expiry = [1.0 if s > 100 else 0.0 for s in spots_fine]
axes[1, 0].plot(spots_fine, dc_vals, 'g-', linewidth=2, label='5d before expiry')
axes[1, 0].plot(spots_fine, dc_expiry, 'k--', linewidth=1, label='At expiry')
axes[1, 0].set_title('Digital Call (K=100, Q=1)')
axes[1, 0].legend()
axes[1, 0].set_ylabel('Price / Payoff')
axes[1, 0].set_xlabel('Spot')

# Digital put payoff
dp_vals = [digital_put(s, 100, T_plot, 0.05, 0.25) for s in spots_fine]
dp_expiry = [1.0 if s < 100 else 0.0 for s in spots_fine]
axes[1, 1].plot(spots_fine, dp_vals, 'm-', linewidth=2, label='5d before expiry')
axes[1, 1].plot(spots_fine, dp_expiry, 'k--', linewidth=1, label='At expiry')
axes[1, 1].set_title('Digital Put (K=100, Q=1)')
axes[1, 1].legend()
axes[1, 1].set_xlabel('Spot')

plt.tight_layout()
plt.savefig('../data/08_exotic_payoffs.png', dpi=100, bbox_inches='tight')
plt.show()


## Desk Reality -- Barrier Placement Is Not Neutral

Here is something textbooks never tell you. When you trade a barrier
option OTC, the dealer knows where your barrier is. The skew around
that strike is priced with that knowledge. In FX especially, barriers
at round numbers (1.2000 EUR/USD, 150.00 USD/JPY) attract hedging
flows that become self-fulfilling.

I have seen barriers defended for days by the flow of the dealer
hedging their exposure -- and then pierced violently on a stop-hunt
when the position got too crowded. The closed-form price above does
not capture any of this. The market microstructure around barriers
is a whole separate discipline.

For more on FX exotics, barrier risk, and the skew dynamics around
barriers, see [FX Options & Structured Products -- A Practitioner's
Approach](https://www.amazon.com/dp/B0H3VSV88X) and the
[working paper](https://doi.org/10.5281/zenodo.20509708).

---

**Next:** [Module 09 -- Heston Stochastic Volatility](09_heston.py)

---
*Djellal Djouad -- CrossVol Research -- 2026*
